# 09_01 The encoder and the decoder: can you train a translator in a few minutes?

The chapter's design is two recurrent networks joined by one vector. In this notebook you build it in
PyTorch, turn a sentence pair into the tensors it trains on, write the one line that is teacher forcing,
and then train the same small translator twice on the same pairs: once with the teacher and once feeding
it its own guesses.

**How this notebook works.** Every notebook in this course has the same rhythm:

1. **Recall.** Answer from memory before you look anything up. `ask()` tells you at once whether you were right.
2. **Predict, then run.** Before a cell with a surprise in it, write your prediction into `guess()`. The next cell runs the code and `reveal()` compares.
3. **Worked example, then your turn.** One example is done in full; the next, near-identical one has lines marked `# YOUR CODE HERE`.
4. **Check.** A `check_...()` cell tests what you saved, exactly as the checkpoint will, and says what to fix.

Run cells in order with **Shift+Enter**. If you get lost, **Kernel, Restart Kernel and Run All Cells** starts clean.

Running this in Google Colab? This cell sets it up; in CourseLabs it does nothing.

In [ ]:
# Colab setup. In a CourseLabs session this cell does nothing.
import os, sys
if "google.colab" in sys.modules:
    import importlib, importlib.util, subprocess
    LAB, REPO = "lab-nlp-09-from-one-language-to-another", "/content/nlp-course"
    if not os.path.isdir(REPO):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/fenago/nlp-course.git", REPO], check=True)
    os.chdir(f"{REPO}/{LAB}")
    if not os.path.exists("data"):
        os.symlink("../data", "data")
    os.makedirs("out", exist_ok=True)
    os.environ["NLPLAB_DATA"] = f"{REPO}/data"
    sys.path.insert(0, os.getcwd())
    PIP = {'torch': 'torch'}
    missing = [spec for mod, spec in PIP.items() if importlib.util.find_spec(mod) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
        importlib.invalidate_caches()
    print(f"Ready: {LAB} and its data are in {os.getcwd()}; installed {len(missing)} package(s).")
elif not os.path.isdir("/opt/nlplab/data") and os.path.isdir("data"):
    # A downloaded copy on your own computer: the helpers read data/ from here.
    os.environ["NLPLAB_DATA"] = os.path.abspath("data")

In [ ]:
import json
import math
import os
import time
import torch
import translate
from nlpcheck import ask, guess, reveal, check_09_01

torch.set_num_threads(4)
data = translate.load_split()
print(len(data["train"]), "training pairs,", len(data["test_src"]), "held-out English sentences")
print(len(data["src_itos"]), "English and", len(data["tgt_itos"]), "Spanish vocabulary entries, four of them special")

## 1. Recall

**r1.** What lets an LSTM remember the start of a long sentence where a plain RNN forgets it? (a) it reads the
sentence twice, (b) a separate cell state that gates add to and erase from, which the gradient can travel
back along, (c) a larger hidden state

**r2.** Which shape of sequence problem is translation? (a) many to one, (b) many to many, in step,
(c) many to many, not in step

In [ ]:
ask("r1", "")
ask("r2", "")

## 2. From a sentence pair to numbers

Each sentence is split into lowercase words and punctuation marks (`translate.tokens`), and each token is
replaced by its position in a vocabulary: the 6,000 commonest English training words and the 8,000
commonest Spanish ones, after four special entries. `<pad>` fills short sentences out to the length of the
longest in a batch, `<unk>` stands for a word too rare to be in the vocabulary, and `<sos>` and `<eos>` mark
where a translation starts and ends. The Spanish side gets them; the English side does not need them.

In [ ]:
english, spanish = data["train"][40000]
print(english, "->", spanish)
print("source ids:", translate.encode(english, data["src_itos"]))
print("target ids:", translate.encode(spanish, data["tgt_itos"], sos=True, eos=True))
print("special ids:", dict(zip(translate.SPECIALS, range(4))))

## 3. The encoder hands on one vector

`translate.Seq2Seq` is the chapter's model with GRUs: an embedding layer and a GRU for each language, and a
linear layer that turns the decoder's state into a score for each of the 8,004 Spanish entries. Read its
`encode`, `decode_step` and `forward` in `translate.py`; each is a few lines. Here is the context vector
for two sentences of different lengths, encoded as one batch:

In [ ]:
torch.manual_seed(0)
model = translate.Seq2Seq()
src = translate.batchify([translate.encode(translate.tokens(s), data["src_itos"])
                          for s in ("Hello.", "I am taking a bath now.")])
print("source batch:", tuple(src.shape), "(sentences, longest)")
print("context:", tuple(model.encode(src).shape), "(layers, sentences, hidden)")
print(sum(p.numel() for p in model.parameters()), "weights in the model")

Two tokens or seven, each sentence arrives at the decoder as the same 256 numbers.

## 4. Your turn: the one line that is teacher forcing

Below is a batch of four target sentences, `tgt`, each `<sos>` ... `<eos>` and padded. With teacher forcing
the decoder **reads** the true words and is scored on **writing** the next one, so it needs two copies of
each target, shifted by one: `tgt_in`, everything but the last token, and `tgt_out`, everything but the
first. Slice the tensor's second dimension to make them.

In [ ]:
src, tgt = next(translate.make_batches(data, data["train"], 4, seed=1))
print(tgt)
tgt_in = None     # YOUR CODE HERE: every target without its last token
tgt_out = None    # YOUR CODE HERE: every target without its first token, <sos>
print("tgt_in:", None if tgt_in is None else tgt_in[0].tolist())
print("tgt_out:", None if tgt_out is None else tgt_out[0].tolist())

Now the untrained model's loss on those four sentences: the cross-entropy of its scores against `tgt_out`,
averaged over every real word. Predict it before you run it. A hint: an untrained model has no preference,
so it spreads its probability roughly evenly over the 8,004 entries.

In [ ]:
guess("untrained_loss", None)   # a number

In [ ]:
logits = model(src, tgt_in)
untrained = torch.nn.functional.cross_entropy(logits.reshape(-1, logits.size(-1)), tgt_out.reshape(-1),
                                              ignore_index=translate.PAD).item()
print(f"untrained loss {untrained:.3f};  ln(8,004) = {math.log(8004):.3f}")
reveal("untrained_loss", round(untrained, 1))

About 9.0, which is ln(8,004). Cross-entropy is minus the log of the probability given to the right word,
and a model that gives every entry the same probability gives each one 1/8,004. That number is the starting
line of every training run in this lab: any loss below 9 is something learned.

## 5. With the teacher, and without

Two copies of a small translator (embeddings of 64, a hidden state of 128) each make one pass over the
29,676 training pairs whose English is at most five tokens, and are scored with chrF on the held-out
sentences of the same length. The first is trained with teacher forcing. The second, with
`teacher_forcing=False`, is fed its own previous guess at every step, as it will be when translating. Same
pairs, same order, same number of steps. Which translates better, "teacher", "own guesses", or is it a
"tie"? The cell takes two to four minutes, most of it the second run, and prints how long each took.

In [ ]:
guess("teacher_wins", None)   # "teacher", "own guesses" or "tie" 

In [ ]:
short = [p for p in data["train"] if len(p[0]) <= 5]
ev_src, ev_refs = translate.eval_split(data)
keep = [i for i, s in enumerate(ev_src) if len(s) <= 5]
ev_src, ev_refs = [ev_src[i] for i in keep], [ev_refs[i] for i in keep]
small = {}
for name, tf in (("teacher", True), ("own guesses", False)):
    torch.manual_seed(0)
    m = translate.Seq2Seq(emb=64, hidden=128)
    t = time.time()
    translate.train(m, data, pairs=short, epochs=1, teacher_forcing=tf)
    seconds = time.time() - t
    small[name] = (m, translate.chrf(translate.greedy(m, ev_src, data), ev_refs), seconds)
    print(f"{name:12} chrF {small[name][1]:.1f}   trained in {seconds:.0f} s")
    for s in ("I am hungry.", "Where is the station?", "Tom is my friend."):
        print("    ", s, "->", translate.translate(m, s, data))
winner = "teacher" if small["teacher"][1] > small["own guesses"][1] + 2 else "own guesses" if small["own guesses"][1] > small["teacher"][1] + 2 else "tie"
reveal("teacher_wins", winner)

Teacher forcing wins, on the same pairs and the same number of steps: in a run measured for this lab, chrF 14
against 10, with "Tom is my friend." coming out as "tom es mi" against "tom es es . .". Each step of the
teacher-forced model learned from a correct history, while the other spent its steps learning to continue
sentences that had already gone wrong, which is why it repeats itself. Now compare the two times. With every
input known in advance, the teacher-forced decoder runs as one `nn.GRU` call over the whole target; fed its
own guesses, it has to wait for each word before it can compute the next, one call per word. On a model this
small the difference is modest and moves with how busy the machine is (runs measured for this lab ranged
from 77 against 83 seconds to 60 against 153); on large models, trained on many processors at once, being
able to compute every position in parallel is most of the reason teacher forcing is universal.

Neither is a translator yet: one pass over short sentences is too little, and chrF 14 is mostly "tom" and the
punctuation. The lab's own translator, twelve times the size and trained for two full passes over all
89,073 pairs, has been training in the background since your session started. The next notebook uses it.

In [ ]:
os.makedirs("out", exist_ok=True)
json.dump({"tgt": tgt.tolist(),
           "tgt_in": None if tgt_in is None else tgt_in.tolist(),
           "tgt_out": None if tgt_out is None else tgt_out.tolist(),
           "untrained_loss": untrained,
           "tf_chrf": small["teacher"][1], "no_tf_chrf": small["own guesses"][1]},
          open("out/09_01_results.json", "w"), indent=1)
check_09_01()

## 6. Exit ticket

**x1.** What is the context vector? (a) the encoder's final hidden state, which becomes the decoder's first
state, (b) the average of the encoder's outputs, (c) the embedding of the first English word

In [ ]:
ask("x1", "")

Explain it back: why can the teacher-forced decoder process a whole target sentence in one call during
training, but not when translating?

*Your explanation:* 